# 使用 OpenAI Agents SDK 和 Amazon Bedrock AgentCore 构建 ChatGPT 插件

在本 cookbook 中，你将为一家虚构的航空公司构建一个私有航班助手，并学习如何将 ChatGPT 连接到以 Amazon Bedrock 为后端的 OpenAI Agents SDK 工作流。这家示例航空公司名为 **Eliza Airlines**；其 `ELZ` 航班使用示例数据，无法预订、改签或取消。

完成本教程后，你将知道如何：

- 向 ChatGPT 暴露只读工具，并验证其输入和输出；
- 通过 OpenAI Secure MCP Tunnel 连接私有工具服务器；
- 使用 Bedrock 模型运行 Agents SDK 工作流，并在 ChatGPT 小组件中显示其结果；
- 在本地检查集成、评估新生成的智能体响应，并验证追踪数据的送达位置。

默认路径是在你的工作站上运行智能体。你将从无需云凭据的检查开始，然后再决定是否使用 Bedrock 运行智能体并将其连接到 ChatGPT。现有的 Amazon Bedrock AgentCore Runtime 是一种可选的托管方案；本 notebook 不会部署它。

由 Eliza 和 OpenAI 联合编写的 cookbook。


## 各组件如何协同工作

**Model Context Protocol (MCP)** 让 ChatGPT 能够发现工具、使用结构化输入调用它们，并接收结构化结果。本仓库中的 **MCP 适配器**就是暴露我们三个航班工具的服务器。它会检查请求、运行智能体，并在将响应返回给 ChatGPT 之前对其进行验证。

![默认请求流程：ChatGPT 通过托管隧道和本地 tunnel-client 将工具调用发送到 MCP 适配器和本地 Agents SDK 智能体。智能体调用 Amazon Bedrock，并将追踪数据导出到 AWS。](images/cookbook-boundaries.svg)

箭头表示工具请求的走向。结果沿同一路径返回，**小组件**（ChatGPT 中的一个小型 HTML 视图）会显示经过验证的结果。在 MCP 中，该结果由 `structuredContent` 字段承载。小组件本身不会调用航班后端。

你工作站上的 `tunnel-client` 会向 OpenAI 托管的隧道端点发起一个出站 HTTPS 连接。该连接让 ChatGPT 能够访问位于 `127.0.0.1:8787/mcp` 的适配器。这个地址是**环回地址（loopback）**：只能在同一台机器上访问。你不需要公开的 MCP URL，也不需要入站防火墙规则。

**OpenAI Agents SDK** 运行 Python 工作流及其函数工具。**Amazon Bedrock** 通过与 OpenAI 兼容的端点提供模型。**AgentCore Observability** 接收追踪数据，这些数据记录了一次运行中的步骤、耗时和错误。**span** 记录追踪中的单个操作，例如一次模型调用或工具调用。

在上面所示的默认路径基础上，还有两个可选项：

- 现有的 **AgentCore Runtime**（AWS 的托管智能体服务）可以代替你的工作站来运行工作流。其所有者需提供该 Runtime 的 **Amazon Resource Name (ARN)**（一种 AWS 资源标识符）以及一个仅具备调用权限的角色。所有者负责部署、凭据和运维。
- `COOKBOOK_TRACING_MODE=dual` 会在 AWS 可观测性之外同时添加 OpenAI Traces。它需要一个单独的 OpenAI Platform 追踪密钥。默认值 `aws` 只导出到 AWS。

你的 Platform 管理员必须预先配置托管隧道，并将其与你的 OpenAI 组织和 ChatGPT 工作区关联。在执行任何需要凭据的步骤之前，请先遵循 [README 权限矩阵](../README.md#enterprise-least-privilege-matrix)。你的团队还负责身份、密钥分发、托管和数据保留。[架构指南](../docs/openai-agentkit-cookbook.md#architecture-boundaries)详细介绍了这些职责。


## 前提条件

你需要具备基本的 Python 和命令行经验，另外还需要：

- Python 3.10 或更高版本；
- [`uv`](https://docs.astral.sh/uv/getting-started/installation/)，用于安装和运行 Python 环境；
- 来自锁定开发环境的 JupyterLab，使用 [README 启动命令](../README.md#optional-notebook-and-developer-checks)启动；
- Node.js 24 或更高版本以及 npm，用于 MCP 适配器和评估工具。

本地检查使用示例数据，无需云凭据。实时步骤需要 [README 权限矩阵](../README.md#enterprise-least-privilege-matrix)中列出的访问权限，并可能产生模型、遥测或 Runtime 费用。

设置单元格会定位仓库并导入 [notebook 辅助函数](../runtime-agent/notebook_helpers.py)。这些辅助函数负责处理子进程超时、有限的输出捕获、凭据过滤、服务清理和评估报告。请求和响应示例则保留在 notebook 中。


In [ ]:
import json
import os
import shutil
import sys
import urllib.request
from pathlib import Path

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "runtime-agent" / "pyproject.toml").exists()
     and (path / "mcp-adapter" / "package.json").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the cookbook repository or notebooks directory.")
RUNTIME_DIR, MCP_DIR = REPO_ROOT / "runtime-agent", REPO_ROOT / "mcp-adapter"
sys.path.insert(0, str(RUNTIME_DIR))

from demo_date import demo_travel_date
from notebook_helpers import (
    live_runtime_environment, local_runtime_service, run,
    run_agent_evaluation, selected_environment,
)

DEMO_TRAVEL_DATE = demo_travel_date()
UV, NODE, NPM = (shutil.which(command) for command in ("uv", "node", "npm"))
if not UV or not NODE or not NPM:
    raise RuntimeError("Install uv and Node.js/npm before executing the notebook.")
print("Repository:", REPO_ROOT)
print("Commit:", run(["git", "rev-parse", "HEAD"], REPO_ROOT).stdout.strip())
for name, command in (("uv", UV), ("Node", NODE), ("npm", NPM)):
    print(name + ":", run([command, "--version"], REPO_ROOT).stdout.strip())


## 1. 安装固定版本的依赖

**锁文件（lockfile）**记录了依赖的确切版本。仓库中包含 `uv.lock` 和两个 npm 锁文件，便于你复现经过测试的环境。如果清单文件与其锁文件不一致，`uv sync --locked` 和 `npm ci` 会失败。


In [ ]:
run([UV, "sync", "--locked", "--extra", "dev"], RUNTIME_DIR)
run([NPM, "ci"], RUNTIME_DIR)
run([NPM, "ci"], MCP_DIR)
print("Locked dependency environments are ready.")

## 2. 跟随一个请求穿过智能体

**schema** 定义了请求或响应可接受的字段和值。[RuntimeRequest](../runtime-agent/schemas.py) 会在任何工具运行之前验证操作及其参数。对于 `search_flights`，它要求提供出发地、目的地和出行日期。

[build_agent](../runtime-agent/agent.py) 将该操作映射到一个函数工具，即通过 Agents SDK 暴露给模型的 Python 函数。它只提供与所请求操作对应的工具，并要求模型调用一次。`agent.py` 中的核心配置如下：

```python
Agent(
    name="ChatGPT MCP flight cookbook agent",
    instructions="Call the selected read-only function tool exactly once and return its JSON.",
    model=model or os.environ.get(MODEL_ENV, DEFAULT_MODEL),
    output_type=AgentOutputSchema(dict, strict_json_schema=False),
    model_settings=ModelSettings(
        parallel_tool_calls=False,
        tool_choice=tool_choice,
    ),
    tools=tools,
    tool_use_behavior="stop_on_first_tool",
)
```

这里，`tools` 包含所选的函数，`tool_choice` 是它的名称。`stop_on_first_tool` 会直接返回工具的输出，而不会让模型重写它。在实际运行中，`build_bedrock_model()` 提供 Bedrock 客户端，`Runner.run_sync()` 执行该智能体。runner 会将返回的 JSON 作为 `RuntimeResponse` 进行验证。

首先，构建一个搜索智能体并在不调用模型的情况下检查其工具。示例日期默认为 UTC 今天加 45 天；可以通过 `COOKBOOK_DEMO_TRAVEL_DATE` 固定它，以便进行可重复的检查。


In [ ]:
from agent import build_agent
from schemas import RuntimeRequest

search_request = RuntimeRequest(
    action="search_flights",
    origin="DAL",
    destination="MDW",
    travel_date=DEMO_TRAVEL_DATE,
)
flight_agent = build_agent(search_request, execution_mode="local")
print("Selected tool:", flight_agent.tools[0].name)
print("After the tool returns:", flight_agent.tool_use_behavior)


现在运行 Python 测试、格式检查和类型检查，然后通过本地入口发送同样的请求。`COOKBOOK_FORCE_LOCAL_TOOLS=1` 会直接调用示例函数，因此这项检查既不使用模型，也不使用 AWS。对于相同的输入，这些函数会返回相同的示例结果。

`executionMode` 标识所选的路由：`local` 或 `deployed`。较旧的 `provider: "agentcore-runtime"` 字段出于兼容性而保留，并不表示智能体在哪里运行。


In [ ]:
run([UV, "run", "pytest", "-q"], RUNTIME_DIR)
run([UV, "run", "ruff", "check", "."], RUNTIME_DIR)
run([UV, "run", "ruff", "format", "--check", "."], RUNTIME_DIR)
run([UV, "run", "pyright"], RUNTIME_DIR)

local_env = selected_environment(overrides={
    "COOKBOOK_FORCE_LOCAL_TOOLS": "1",
    "COOKBOOK_EVENT": search_request.model_dump_json(exclude_none=True),
})
smoke = run([UV, "run", "python", "agent.py"], RUNTIME_DIR, local_env)
local_result = json.loads(smoke.stdout)
assert local_result["provider"] == "agentcore-runtime"
assert local_result["executionMode"] == "local"
assert local_result["action"] == "search_flights"
assert local_result["data"]["flights"]
assert all(flight["travelDate"] == DEMO_TRAVEL_DATE for flight in local_result["data"]["flights"])
print(json.dumps(local_result, indent=2))


## 3. 在本地调用 Runtime HTTP 接口

AgentCore Python SDK 提供两个 HTTP 端点：`/ping` 报告健康状态，`/invocations` 接收智能体请求。辅助函数会在 8080 端口启动该服务器，并在单元格结束时（包括失败时）停止它。下面的请求从搜索结果中取出一个航班，并使用相同的日期检查其状态。

这会在你的工作站上使用示例工具测试已部署 Runtime 的接口。其入口会将响应标记为 `executionMode: "deployed"`；仅凭这个标签并不能证明已部署到 AWS。


In [ ]:
first_flight = local_result["data"]["flights"][0]
with local_runtime_service(UV, RUNTIME_DIR) as health:
    request = urllib.request.Request(
        "http://127.0.0.1:8080/invocations",
        data=json.dumps({
            "action": "get_live_status",
            "flight_number": first_flight["flightNumber"],
            "origin": first_flight["origin"],
            "destination": first_flight["destination"],
            "travel_date": first_flight["travelDate"],
        }).encode(),
        headers={"content-type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=10) as response:
        invocation = json.loads(response.read())
    assert health["status"] == "Healthy"
    assert invocation["executionMode"] == "deployed"
    assert invocation["action"] == "get_live_status"
    assert invocation["data"]["flight"]["flightNumber"] == "ELZ1234"
    assert invocation["data"]["flight"]["travelDate"] == DEMO_TRAVEL_DATE
    print("Health:", health)
    print("Invocation:", json.dumps(invocation, indent=2))


## 4. 检查 MCP 适配器和小组件

适配器通过 **Streamable HTTP**（一种通过 HTTP 交换消息的 MCP 传输方式）暴露 `search_flights`、`get_upcoming_status` 和 `get_live_status`。每个工具都接受已定义的输入 schema，并返回特定于操作的输出 schema。这些工具是**幂等的**：重复调用不会改变示例数据。

TypeScript 测试覆盖了工具发现、调用、响应验证，以及使用模拟 AWS 发送器的 AWS 调用代码。它们还会检查小组件的带版本资源 URI、内容类型（MIME 类型）和**内容安全策略（Content Security Policy，CSP）**，后者限制小组件可以加载的资源。小组件使用 `textContent` 显示结果文本，而不会将其当作 HTML 处理。这些测试不会进行实时 AWS 调用。

关键边界位于 [server.ts](../mcp-adapter/server.ts)：适配器将经过验证的数据作为 `structuredContent` 返回，[flight-widget.html](../mcp-adapter/public/flight-widget.html) 在收到 `ui/notifications/tool-result` 时对其进行渲染。


In [ ]:
run([NPM, "run", "typecheck"], MCP_DIR)
run([NPM, "test"], MCP_DIR)
run([NPM, "run", "build"], MCP_DIR)

widget = (MCP_DIR / "public" / "flight-widget.html").read_text(encoding="utf-8")
assert "ui/notifications/tool-result" in widget
assert "textContent" in widget
assert "<script src=" not in widget
print("Typed MCP server and self-contained widget validated.")

## 5. 评估响应并验证追踪

**评估**检查输出是否符合预期。**追踪**记录执行期间发生了什么。本 cookbook 使用 Promptfoo 为响应打分，使用 AgentCore Observability 检查运行情况；这是两项相互独立的检查。

![Promptfoo 在发布前为新生成的智能体响应打分。AWS 默认接收追踪数据，在 dual 模式下还可使用 OpenAI Traces。一条可选的 AWS 评估路径会为专用 Runtime 已完成的 span 打分。](images/tracing-evaluation-paths.svg)

[Promptfoo](../runtime-agent/promptfooconfig.agent.cjs) 将三个航班场景发送到本地 Agents SDK 工作流，然后检查每个响应的结构和预期行为。**AWS Distro for OpenTelemetry (ADOT)** 对智能体进行插桩，以记录 span 并将其导出到 AWS。`COOKBOOK_TRACING_MODE=aws` 是默认值；`dual` 还会使用单独的 `OPENAI_TRACE_API_KEY` 将追踪数据发送到 OpenAI。

先运行 `eval:agent:validate`，它会在不调用智能体的情况下检查评估配置。另一项无需凭据的检查是 `eval:run`，它通过一个原样返回输入的 echo provider 回放已存储的正例和反例（称为 **fixtures**）。这会用示例响应来检查断言。

要评估新生成的智能体响应，请在启动 Jupyter 之前设置 `RUN_PROMPTFOO_AGENT_EVALUATION=1`。你可以通过 `PROMPTFOO_AGENT_EVALUATION_CASE_IDS` 选择较少的用例。辅助函数会运行受保护的命令并汇总新报告。报告保存在被忽略的 `runtime-agent/evals/results/` 目录中；Promptfoo 的分享、遥测和远程生成功能均已禁用。真实的智能体运行可能会产生模型和遥测费用。

报告通过并不能确认追踪数据已送达。请运行追踪冒烟测试命令，并将其输出的**关联 ID（correlation ID）**（同一次调用的各条记录共享的标识符）用于 [AWS 追踪检查器](../docs/tracing-and-publication.md#trace-verification-is-a-separate-step)。请考虑数据摄取的延迟。在 dual 模式下，验证者还必须在 OpenAI Traces 中找到匹配的 ID；本 notebook 不使用未公开文档的追踪查询 API。

[AgentCore Evaluations](../docs/aws-evaluation.md) 提供了一个可选的 AWS 原生替代方案。它会调用一个专用的非生产 Runtime，等待 CloudWatch 中的 span 完成，然后使用内置评估器为其打分。这些评估器需要提示词、响应和工具内容，因此 Runtime 所有者必须在那里批准并启用内容捕获。评估与普通的 ChatGPT 请求分开运行。


In [ ]:
run([NPM, "run", "eval:agent:validate"], RUNTIME_DIR)
print("Promptfoo actual-agent configuration validated.")

if os.getenv("RUN_PROMPTFOO_AGENT_EVALUATION", "0") == "1":
    evaluation_summary = run_agent_evaluation(NODE, RUNTIME_DIR)
    print(json.dumps(evaluation_summary, indent=2))
else:
    print("Skipped fresh agent runs. Set RUN_PROMPTFOO_AGENT_EVALUATION=1 before starting Jupyter.")


## 6. 可选：调用现有的 AWS Runtime

如果你走的是默认的本地路径，请跳过这一步。对于现有的 Runtime，此单元格会通过 AWS SDK 发起两次真实的 `InvokeAgentRuntime` 调用：一次 DAL 到 MDW 的搜索，然后对第一个结果进行状态检查。它会验证两个响应，并检查航班和日期保持一致。此单元格的输出中省略了追踪和会话标识符。

在启动 Jupyter 之前，请设置 `RUN_LIVE_AWS=1`，以及由 Runtime 所有者提供的 `AWS_PROFILE`、`AGENTCORE_RUNTIME_AGENT_ARN` 和 `AGENTCORE_RUNTIME_REGION`。[README 启动命令](../README.md#optional-notebook-and-developer-checks)会加载根目录下的私有 `.env`，并保留由你的凭据进程导出的值。更改配置后请重启内核。[权限矩阵](../README.md#enterprise-least-privilege-matrix)列出了所需的仅调用权限。


In [ ]:
if os.getenv("RUN_LIVE_AWS", "0") == "1":
    if not os.getenv("AGENTCORE_RUNTIME_AGENT_ARN", "").strip():
        raise RuntimeError("Set AGENTCORE_RUNTIME_AGENT_ARN before enabling RUN_LIVE_AWS.")
    live_env = live_runtime_environment(DEMO_TRAVEL_DATE)
    # Call the built runner directly to keep unrelated .env credentials out.
    live = run([NODE, "dist/live-smoke.js"], MCP_DIR, live_env, timeout_seconds=300)
    live_result = json.loads(live.stdout.strip().splitlines()[-1])
    assert live_result["provider"] == "agentcore-runtime"
    assert live_result["executionMode"] == "deployed"
    assert live_result["action"] == "get_live_status"
    assert live_result["flight"]["flightNumber"] == "ELZ1234"
    assert live_result["flight"]["travelDate"] == DEMO_TRAVEL_DATE
    print(json.dumps(live_result, indent=2))
else:
    print("Skipped live AWS invocation. Set RUN_LIVE_AWS=1 after configuring a sandbox runtime.")


## 7. 通过 Secure MCP Tunnel 在 ChatGPT 中测试插件连接

本地检查通过后，请参照 [README 连接演练](../README.md#7-connect-the-mcp-server-to-chatgpt)。如果你选择了现有的 Runtime，请先完成第 6 步。

1. 在 `mcp-adapter/` 中运行 `npm start` 启动私有 MCP 服务器，它监听 `http://127.0.0.1:8787/mcp`.
2. 确认 Platform 管理员已创建托管隧道，将其与正确的 OpenAI 组织和 ChatGPT 工作区关联，并向你提供了完整的隧道 ID 和一个具有 **Tunnels Read + Use** 权限的运行时密钥。然后使用该隧道 ID 和环回 MCP URL 配置一个 OpenAI `tunnel-client` profile。
3. 运行 `tunnel-client doctor --profile agentcore-cookbook --explain`，然后运行 `tunnel-client run --profile agentcore-cookbook`，并等待 `http://127.0.0.1:8080/readyz`.
4. 在打开 [ChatGPT Plugins](https://chatgpt.com/plugins) 之前，先在 **Settings → Security and login** 中启用开发者模式。如果该设置或 Plugins 页面不可用，请联系工作区管理员检查工作区策略。
5. 在 Plugins 页面上，点击加号按钮，输入面向用户的名称和描述，然后在 **Connection** 下选择 **Tunnel**，并选择或粘贴完整的隧道 ID。
6. 选择 **No authentication**。隧道运行时密钥用于让 `tunnel-client` 向隧道服务进行身份验证；不要将其粘贴到插件表单中。创建插件连接，并等待自动工具发现列出 `search_flights`、`get_upcoming_status` 和 `get_live_status`。
7. 在新对话中从工具菜单添加该插件连接，然后测试搜索、即将起飞航班状态和实时状态的提示词。
8. 确认预订请求无法选择任何写入工具。

隧道客户端以出站方式建立连接。请将 MCP 服务器保持在环回地址上；不要公开 8787 端口，也不要添加入站防火墙规则。这一步测试的是 ChatGPT 中的连接；它不会验证其他 OpenAI 产品界面。

测试成功意味着你拥有了一个可用的私有插件连接。在公开发布之前，你的团队仍需安排公共服务端点，并解决运营和发布责任归属、许可、隐私和安全审查，以及任何所需的提交审批。仓库的 MIT 许可证涵盖代码本身，但并不提供发布批准。请参阅[发布要求](../docs/tracing-and-publication.md#private-testing-is-not-public-distribution)。


## 总结

你已经跟随一个航班请求，依次经过了经验证的 MCP 输入、Agents SDK 工作流，以及在 ChatGPT 中显示结果的小组件。该示例展示了私有工具服务器如何在使用 Bedrock 模型的同时，将凭据隔离在小组件之外。

本地检查无需云凭据即可验证示例逻辑和集成契约。可选的实时步骤则让你能够在自己的环境中评估新生成的智能体响应、验证追踪数据的送达，并测试私有 ChatGPT 连接。

要改造此示例：

1. 将 [tools.py](../runtime-agent/tools.py) 中的示例函数替换为对经批准服务的只读调用，例如库存或订单状态 API。
2. 同步更新 [Python schemas](../runtime-agent/schemas.py)、[MCP 工具注册](../mcp-adapter/server.ts)、[MCP 响应 schemas](../mcp-adapter/schemas/flight.ts) 和[小组件](../mcp-adapter/public/flight-widget.html)。保留严格验证、只读注解和凭据隔离。
3. 向[评估 fixtures](../runtime-agent/evals/fixtures/flight-status-results.jsonl) 中添加有代表性的成功用例、格式错误的输入和不支持的操作。在评估真实智能体之前先运行本地检查。
4. 配置经批准的 Bedrock 模型、隧道和追踪目标。你的团队可以让智能体保持在本地运行，也可以提供现有的 Runtime 进行托管执行。

测试完成后，请停止 MCP 服务器和隧道客户端；请参阅[操作说明](../README.md#10-everyday-operation)。[架构指南](../docs/openai-agentkit-cookbook.md)和 [AWS IAM 指南](../docs/aws-iam.md)介绍了为你自己的服务做出托管和访问决策时需要考虑的内容。


## 贡献者

本 cookbook 由 OpenAI 和 Eliza 联合合作完成。

- [Rohan Awasthi](https://www.linkedin.com/in/rohan-awasthi-4a5208233/) (Eliza)
- [Chuck Hernandez](https://www.linkedin.com/in/chuck-hernandez/) (Eliza)
- [Syed Ahmed](https://www.linkedin.com/in/syed-a-usanaemeaapeastwest/) (OpenAI)
- [Steven McAteer](https://www.linkedin.com/in/steven-mcateer/) (Eliza)